# Practical 3: Pixel-Level Spatial GeoAI and GPS Drone Scouting
### Hands-on Session: Pinpointing Anomaly Coordinates, Plot IDs, and Spatial Hotspots with Google Gemini

In this practical:
- Ingest spatial anomaly telemetry derived from the 30-meter pixel raster cube.
- Feed exact Plot IDs, bad patch sectors, and GPS coordinates into Google Gemini.
- Generate a precision Field Scouting and Ground Action Plan.
- Ask targeted drone scouting and trouble-spot questions cell-by-cell.

## Step 1: Import required libraries

In [1]:
import os
import sys
import json
import pandas as pd
from IPython.display import display, Markdown
from dotenv import load_dotenv
from google import genai

## Step 2: Initialize Google GenAI Client with your API key from .env

In [2]:
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
model_name = os.getenv("GEOAI_MODEL", "gemini-3.5-flash-lite")

client = genai.Client(api_key=gemini_api_key)
print("Google GenAI Client successfully initialized.")
print("Target Model:", model_name)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Google GenAI Client successfully initialized.
Target Model: gemini-3.5-flash-lite


## Step 3: Ingest pixel-level spatial anomaly telemetry
Loading pre-computed spatial anomalies containing Plot IDs, problem sectors, and exact GPS coordinates.

In [3]:
df_scout = pd.read_parquet("data/rice_farm_scout_anomalies.parquet")
scout_data = df_scout[['plot_id', 'total_pixels', 'bad_pixel_count', 'bad_pixel_percent', 'problem_sector', 'worst_spot_latitude', 'worst_spot_longitude']].head(6).to_dict(orient='records')

print("Pixel-Level Spatial Anomaly Telemetry:")
print(json.dumps(scout_data, indent=2))

Pixel-Level Spatial Anomaly Telemetry:
[
  {
    "plot_id": "Plot_01",
    "total_pixels": 381,
    "bad_pixel_count": 44,
    "bad_pixel_percent": 11.5,
    "problem_sector": "Southern-Eastern corner",
    "worst_spot_latitude": 7.88312,
    "worst_spot_longitude": 8.32974
  },
  {
    "plot_id": "Plot_05",
    "total_pixels": 257,
    "bad_pixel_count": 36,
    "bad_pixel_percent": 14.0,
    "problem_sector": "Southern-Eastern corner",
    "worst_spot_latitude": 7.88929,
    "worst_spot_longitude": 8.3386
  },
  {
    "plot_id": "Plot_15",
    "total_pixels": 332,
    "bad_pixel_count": 32,
    "bad_pixel_percent": 9.6,
    "problem_sector": "Southern-Western corner",
    "worst_spot_latitude": 7.88446,
    "worst_spot_longitude": 8.33215
  },
  {
    "plot_id": "Plot_02",
    "total_pixels": 366,
    "bad_pixel_count": 29,
    "bad_pixel_percent": 7.9,
    "problem_sector": "Southern-Eastern corner",
    "worst_spot_latitude": 7.8842,
    "worst_spot_longitude": 8.33135
  },
  {
   

## Step 4: Generate Live Field Scouting and Ground Action Plan
Google Gemini translates pixel anomalies into practical field scouting instructions with exact Plot IDs and GPS waypoints.

In [4]:
prompt = f"""You are an expert Chief Farm Agronomist briefing a rice farmer and drone scout on where their field workers must walk today.
Do not use technical math jargon or decimal index numbers. Talk about practical farm reality (stunted seedlings, water pools, weed patches, soil leveling).

Here are the bad spots found by satellite pixels in the plots:
{json.dumps(scout_data, indent=2)}

Write a direct Field Scouting Action Plan:
1. Priority Plots with Bad Patches: Clearly name the top 3 plots with trouble spots (mention Plot ID and which corner or sector of the field it is in).
2. What the Farm Scout Should Look For on the Ground: For each trouble spot, explain what physical problem the worker is likely to see (e.g. stagnant standing water from poor leveling, yellowing from nutrient loss, or weed choke).
3. Exact GPS Coordinates: Give the coordinates as clear GPS waypoints that the farmer can paste into a phone or give to a scout.
4. Immediate Ground Remedy: Give simple instructions on what the worker should do (e.g. dig a small channel to drain trapped water, spot-spray weeds, or spot-apply a handful of fertilizer).

Format in clean Markdown with clear headings and bold Plot IDs. Do not use emojis."""

print("Calling Google Gemini GeoAI API for spatial scout report...")
response = client.models.generate_content(model=model_name, contents=prompt)
display(Markdown(response.text))

Calling Google Gemini GeoAI API for spatial scout report...


# Field Scouting Action Plan

Good morning, team. The latest satellite scans have highlighted several trouble spots across our rice fields that need immediate ground-level investigation today. We have a clear pattern of stress in specific corners of our plots. 

Review the scouting route, target problems, and immediate remedies below before sending the field workers out.

---

### 1. Priority Plots with Bad Patches

We are focusing our ground crew on the top three plots showing the highest concentration of crop stress:

*   **Plot_05**: Heavy stress concentrated in the **Southern-Eastern corner**.
*   **Plot_01**: Widespread stress showing up in the **Southern-Eastern corner**.
*   **Plot_15**: Noticeable crop slowing in the **Southern-Western corner**.

---

### 2. What the Farm Scout Should Look For on the Ground

When the field workers reach these specific corners, they should inspect the crop and soil for these physical issues:

*   **Plot_05 (Southern-Eastern corner):** Look for deep, stagnant standing water caused by poor soil leveling. The rice seedlings here are likely drowning, turning yellow, and showing signs of root rot due to a lack of oxygen.
*   **Plot_01 (Southern-Eastern corner):** Expect to find aggressive weed patches choking out the young rice plants, combined with pale, stunted seedlings suffering from nutrient loss.
*   **Plot_15 (Southern-Western corner):** Check for high spots in the field where water is running off too quickly, leaving the soil dry and cracking, which stunts early seedling growth.

---

### 3. Exact GPS Coordinates

Provide these exact waypoints to your drone scout and field workers so they can navigate directly to the worst spots in each plot:

*   **Plot_05 Target:** Latitude 7.88929, Longitude 8.33860
*   **Plot_01 Target:** Latitude 7.88312, Longitude 8.32974
*   **Plot_15 Target:** Latitude 7.88446, Longitude 8.33215
*   **Plot_02 Target:** Latitude 7.88420, Longitude 8.33135
*   **Plot_06 Target:** Latitude 7.89117, Longitude 8.32195
*   **Plot_12 Target:** Latitude 7.90136, Longitude 8.32436

---

### 4. Immediate Ground Remedy

Instruct the field workers to carry out these practical fixes as soon as they locate the problems:

*   **For trapped water (Plot_05 and other wet spots):** Manually dig a small drainage channel with a shovel to let the pooled water escape into the main ditch. Do not let water sit and scald the seedlings.
*   **For weed choke (Plot_01):** Pull out the heavy weed clusters by hand immediately, or mark the patch for targeted spot-spraying if the infestation is too dense for manual removal.
*   **For nutrient-deficient or stunted pockets:** Apply a quick handful of supplemental nitrogen fertilizer directly to the pale, struggling clumps to help them catch up with the rest of the field.

## Step 5: Define interactive Q&A helper for trouble spot queries

In [5]:
def ask_scout_ai(question):
    p = f"""You are an experienced Rice Farm Operations Expert talking to a practical commercial rice farmer.
Always mention the specific Plot IDs, physical sectors (like southern corner, eastern side), and GPS coordinates where appropriate.
Do not use cryptic mathematical formulas or decimal index numbers. Give direct, realistic farming guidance.
Field telemetry: {json.dumps(scout_data)}

Farmer asks: "{question}"
Answer directly and practically in 2-3 short paragraphs. Do not use emojis."""
    r = client.models.generate_content(model=model_name, contents=p)
    return r.text

print("ask_scout_ai function ready.")

ask_scout_ai function ready.


## Step 6: Trouble Spot Question 1: Finding the Worst Bad Patch
*Which specific plot has the worst patch of sick or stunted crops, and which corner should my farm workers walk to?*

In [6]:
q1 = "Which specific plot has the worst patch of sick or stunted crops, and which corner should my farm workers walk to?"
print("Farmer asks:", q1)
display(Markdown(ask_scout_ai(q1)))

Farmer asks: Which specific plot has the worst patch of sick or stunted crops, and which corner should my farm workers walk to?


Plot 05 has the worst overall patch of stunted crops on the entire farm, showing significant stress across fourteen percent of its canopy. Send your crew straight to the southern-eastern corner of that plot. Have them walk directly to GPS coordinates latitude 7.88929, longitude 8.3386 to inspect the root zone and check for localized drainage failure or nutrient deficiency.

Your second priority is Plot 01, where about eleven percent of the crop is struggling in the southern-eastern corner as well. Direct your workers to inspect the worst spot there at latitude 7.88312, longitude 8.32974. Look closely at these specific zones for stem borer damage or uneven fertilizer spread during top-dressing.

## Step 7: Trouble Spot Question 2: Exact Phone GPS Waypoints
*Give me the exact phone GPS waypoints I can give to my tractor driver or field scout to find the weak spots immediately.*

In [7]:
q2 = "Give me the exact phone GPS waypoints I can give to my tractor driver or field scout to find the weak spots immediately."
print("Farmer asks:", q2)
display(Markdown(ask_scout_ai(q2)))

Farmer asks: Give me the exact phone GPS waypoints I can give to my tractor driver or field scout to find the weak spots immediately.


Here are the exact coordinates for your field scout and tractor driver to locate the weak spots across your struggling plots. Send these straight to their phones.

The highest priority is **Plot_05**, which is showing the worst stress at latitude 7.88929, longitude 8.3386 in the southern-eastern corner. Right nearby in **Plot_01**, tell them to check latitude 7.88312, longitude 8.32974, also in the southern-eastern sector. For **Plot_02**, the trouble spot is just a short distance away at latitude 7.8842, longitude 8.33135 in that same southern-eastern zone. 

Move next to **Plot_15**, where the crew needs to inspect the southern-western corner at latitude 7.88446, longitude 8.33215. For **Plot_06**, send them back to the southern-eastern corner at latitude 7.89117, longitude 8.32195. Finally, check the northern-western corner of **Plot_12** at latitude 7.90136, longitude 8.32436. Have them check these exact drops for water pooling, nutrient deficiency, or early pest pressure.